In [ ]:
import os
from dotenv import load_dotenv
import requests
import time
import math
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
from datasets import load_dataset, Dataset
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

In [ ]:
# loading the github token to increase the rate limits
load_dotenv()
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

In [ ]:
# creating headers with said token
headers = {"Authorization": f"token {GITHUB_TOKEN}"}

In [ ]:
# updating the session with headers
session = requests.Session()
session.headers.update(headers)

In [ ]:
def fetch_issues(
        owner="huggingface",
        repo="datasets",
        num_issues=10000,
        issues_path=Path("data"),
):
    """
    A function to fetch all the issues
    (including pull requests) from a github repo
    """
    if not issues_path.is_dir():
        issues_path.mkdir(exist_ok=True)

    all_issues = []
    per_page = 100
    num_pages = math.ceil(num_issues / per_page)
    base_url = "https://api.github.com/repos"

    for page in tqdm(range(1, num_pages + 1)):
        query = f"issues?page={page}&per_page={per_page}&state=all"
        response = requests.get(f"{base_url}/{owner}/{repo}/{query}", headers=headers)

        remaining = int(response.headers.get("X-RateLimit-Remaining", 1))

        # if the rate limit is hit
        if remaining == 0:
            reset_time = int(response.headers.get("X-RateLimit-Reset", time.time() + 60))
            sleep_seconds = max(reset_time - time.time(), 1) + 1
            print(f"Reached GitHub rate limit. Sleeping for {sleep_seconds:.0f}s...")
            time.sleep(sleep_seconds)
            response = requests.get(f"{base_url}/{owner}/{repo}/{query}", headers=headers)

        data = response.json()
        if not data:
            break

        all_issues.extend(data)

    # creating and saving the datases
    df = pd.DataFrame.from_records(all_issues)
    output_file = f"{issues_path}/{repo}-issues.jsonl"
    df.to_json(output_file, orient="records", lines=True)
    print(
        f"Downloaded {len(all_issues)} issues for {repo}! Dataset stored at {output_file}."
    )

In [ ]:
# fetch_issues()

In [ ]:
# loading the datases
issues_dataset = load_dataset(
    "json",
    data_files="datasets-issues.jsonl",
    split="train",
    chunksize=100*1024*1024 # increasing chunksize, so the loading don't brake of different patterns
)
issues_dataset

In [ ]:
# marking pull requests
issues_dataset = issues_dataset.map(
    lambda x: {"is_pull_request": False if x["pull_request"] is None else True}
)

In [ ]:
def fetch_one(row):
    """
    A function that fetches comments
    for an issue in the dataset.
    """
    if row["comments"] == 0:
        return []
    url = f"https://api.github.com/repos/huggingface/datasets/issues/{row['number']}/comments"
    response = session.get(url)

    # if rate limit is hit
    if response.status_code in (403, 429):
        reset_time = int(response.headers.get("X-RateLimit-Reset", time.time() + 60))
        sleep_seconds = max(reset_time - time.time(), 1) + 1
        print(f"Reached GitHub rate limit. Sleeping for {sleep_seconds:.0f}s...")
        time.sleep(sleep_seconds)
        response = session.get(url)

    data = response.json()
    return [r["body"] for r in data] if isinstance(data, list) else []

In [ ]:
# fetching comments with ThreadPool because the process itself is too slow
rows = [{"number": n, "comments": c} for n, c in zip(issues_dataset["number"], issues_dataset["comments"])]

with ThreadPoolExecutor(max_workers=16) as executor:
    all_comments = list(tqdm(executor.map(fetch_one, rows), total=len(rows)))

In [ ]:
# adding comments to the dataset
issues_with_comments_dataset = issues_dataset.remove_columns("comments").add_column("comments", all_comments)

In [ ]:
# saving the dataset
issues_with_comments_dataset.to_json("datasets-issues-with-comments.jsonl")